# Calibration of the variational mixed-model fit

The Methods state that the mean-field variational fit of the trial-level mixed model was unreliable at
this sample size, and that all reported trial-level estimates therefore come from the MCMC posterior.
This notebook produces the evidence behind that sentence:

1. **False-positive rate.** How often the variational fit's 95% interval for the colour x medication
   coefficient excludes zero on datasets where the true medication effect is zero by construction.
2. **Width.** How the variational posterior SD on the real data compares with the MCMC posterior SD
   from the reported fits.

### Null datasets

Each patient has one OFF and one ON session. Under the null of no medication effect the two labels are
exchangeable within a patient, so randomly swapping them removes any medication effect while keeping
everything else: trial counts, each patient's ability, the session structure and the clustering of
trials within patients. A calibrated method should find an effect in about 5% of such datasets.

### Three specifications

| Spec | Model | Random effects | Role |
|---|---|---|---|
| A | `coh + col*med` | intercept, col, med, col:med | The model originally published (p = 7.5e-9) |
| B | `coh + col*med + prev_rc*med` | intercept, col, med, col:med | The specification the first calibration used |
| C | `coh + col*med + prev_rc*med` | all seven terms | Same specification as the reported MCMC model |

Spec C is the one the Methods sentence describes. Spec B reproduces the earlier calibration run.
Spec A documents what happened to the published estimate.

This notebook writes nothing to disk. It reads the cached MCMC draws in `mcmc_fits/` but never
refits or re-caches them.


In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2
from imports import *


In [2]:
import contextlib
import io
import os
from concurrent.futures import ProcessPoolExecutor

from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
from threadpoolctl import threadpool_limits

from config import dir_config
from src.utils import classification_utils
from src.utils.mcmc_cache import CACHE_DIR, TERMS, prepare_trials

processed_dir = Path(dir_config.data.processed)
filtered_data = pd.read_csv(Path(processed_dir, "processed_all_data_accu_60_filtered.csv"), index_col=None)
processed_metadata = pd.read_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), index_col=None)
subjects_map, _ = classification_utils.get_subject_classification_ids(processed_metadata)

N_PERMUTATIONS = 300
SEED = 0
N_WORKERS = max(1, min(28, (os.cpu_count() or 2) - 2))
SUBTYPES = {"tremor": "tremor_dominant", "bradykinetic": "bradykinesia_dominant"}


PD subjects with ON+OFF sessions: 41
PD subjects with UPDRS subtype: 22
All PD subjects:           41
  Tremor dominant:         11 	 ['P3' 'P6' 'P7' 'P11' 'P12' 'P17' 'P18' 'P19' 'P29' 'P31' 'P32']
  Brady dominant:          10 	 ['P1' 'P4' 'P9' 'P13' 'P20' 'P22' 'P23' 'P28' 'P33' 'P34']
  Intermediate:            1 	 ['P24']
HC subjects:               18 	 ['HC1' 'HC3' 'HC6' 'HC7' 'HC8' 'HC9' 'HC12' 'HC13' 'AV' 'BC' 'BF' 'EM'
 'ES' 'GF' 'GP' 'JA' 'MRM' 'SY']


## Data

Specs B and C use `prepare_trials` from `mcmc_cache`, so they see exactly the trials the reported MCMC
fits see. That function drops the first trial of each session, which has no previous trial. Spec A
has no history term, so like the published model it keeps every trial.


In [3]:
def prepare_without_history(trial_data, subject_ids):
    """Published model's data: every OFF/ON trial, no history term, same coding as mcmc_cache."""
    df = trial_data[trial_data["subject_id"].isin(subject_ids) & trial_data["medication"].isin(["off", "on"])].copy()
    df["coh"] = df["signed_coherence"] / 100
    df["col"] = np.where(df["color"] == 1, 0.5, -0.5)
    df["med"] = np.where(df["medication"] == "on", 0.5, -0.5)
    return df.reset_index(drop=True)


data = {}
for subtype, key in SUBTYPES.items():
    ids = subjects_map[key]
    data[(subtype, "history")] = prepare_trials(filtered_data, ids).assign(subj=lambda x: x["subject_id"])
    data[(subtype, "no_history")] = prepare_without_history(filtered_data, ids).assign(subj=lambda x: x["subject_id"])

# every spec must see the subjects in the same order, so one flip vector means the same null dataset
for subtype in SUBTYPES:
    a = list(data[(subtype, "history")]["subject_id"].unique())
    b = list(data[(subtype, "no_history")]["subject_id"].unique())
    assert a == b, subtype
    print(f"{subtype:13s} {len(a)} subjects | trials with history: {len(data[(subtype, 'history')]):,} "
          f"| without: {len(data[(subtype, 'no_history')]):,}")


tremor        11 subjects | trials with history: 13,518 | without: 13,540
bradykinetic  10 subjects | trials with history: 12,529 | without: 12,549


## Model specifications


In [4]:
VC_FOUR = {"a": "0+C(subj)", "b": "0+C(subj):col", "c": "0+C(subj):med", "d": "0+C(subj):col:med"}
VC_SEVEN = dict(VC_FOUR, e="0+C(subj):coh", f="0+C(subj):prev_rc", g="0+C(subj):prev_rc:med")

SPECS = {
    "A: published": ("choice ~ coh + col*med", VC_FOUR, "no_history"),
    "B: first calibration": ("choice ~ coh + col*med + prev_rc*med", VC_FOUR, "history"),
    "C: matches MCMC": ("choice ~ coh + col*med + prev_rc*med", VC_SEVEN, "history"),
}


def vb_interaction(spec, subtype, flips=None):
    """Variational posterior mean and SD of col:med; `flips` swaps each subject's OFF/ON labels."""
    formula, vc, which = SPECS[spec]
    frame = data[(subtype, which)]
    if flips is not None:
        subjects = frame["subject_id"].unique()
        index = frame["subject_id"].map({s: i for i, s in enumerate(subjects)}).to_numpy()
        frame = frame.assign(med=frame["med"].to_numpy() * flips[index])
    with threadpool_limits(1), contextlib.redirect_stdout(io.StringIO()):
        fit = BinomialBayesMixedGLM.from_formula(formula, vc, frame).fit_vb(verbose=False)
    names = list(fit.model.exog_names)
    return fit.fe_mean[names.index("col:med")], fit.fe_sd[names.index("col:med")]


## Null datasets

One flip vector per permutation, drawn in the same order as the first calibration run (seed 0,
restarted for each subtype), so spec B reproduces that run exactly and all three specs are evaluated
on the same 300 null datasets.


In [5]:
flips = {}
for subtype in SUBTYPES:
    rng = np.random.default_rng(SEED)
    n_subjects = data[(subtype, "history")]["subject_id"].nunique()
    flips[subtype] = np.array([rng.choice([-1.0, 1.0], size=n_subjects) for _ in range(N_PERMUTATIONS)])
    print(f"{subtype:13s} {flips[subtype].shape}  distinct null datasets: {len(np.unique(flips[subtype], axis=0))}")


tremor        (300, 11)  distinct null datasets: 283
bradykinetic  (300, 10)  distinct null datasets: 257


With 11 and 10 patients there are only 2,048 and 1,024 possible label assignments, so some of the 300
draws repeat. Repeats are valid draws from the permutation distribution and are kept, which is also
what the first calibration did.

## Refit on every null dataset

About 1,800 variational fits, run in parallel. Expect a few minutes.


In [6]:
def _task(args):
    spec, subtype, i = args
    mean, sd = vb_interaction(spec, subtype, flips[subtype][i])
    return spec, subtype, i, mean, sd


tasks = [(spec, subtype, i) for spec in SPECS for subtype in SUBTYPES for i in range(N_PERMUTATIONS)]
with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    null_rows = list(tqdm(pool.map(_task, tasks, chunksize=4), total=len(tasks), desc="variational refits"))

null = pd.DataFrame(null_rows, columns=["spec", "subtype", "permutation", "mean", "sd"])
null["z"] = null["mean"] / null["sd"]
null["interval excludes 0"] = null["z"].abs() > stats.norm.ppf(0.975)
print(f"{len(null):,} fits, {null['sd'].isna().sum()} failed")


variational refits: 100%|██████████| 1800/1800 [05:37<00:00,  5.33it/s]

1,800 fits, 0 failed


### Control: the permutation scheme itself

If the null datasets are built correctly, a method known to be calibrated should reject about 5% of
them. The subject-level signed-rank test on each patient's difference-in-differences is that control.
Swapping a patient's labels simply negates their difference-in-differences.


In [7]:
def subject_did(frame):
    zero = frame[frame["coherence"] == 0]
    prior = zero.groupby(["subject_id", "med", "col"])["choice"].mean().unstack("col")
    prior = (prior[0.5] - prior[-0.5]).unstack("med")
    return (prior[0.5] - prior[-0.5]).reindex(frame["subject_id"].unique())


control_rows = []
for subtype in SUBTYPES:
    did = subject_did(data[(subtype, "no_history")]).to_numpy()
    for i in range(N_PERMUTATIONS):
        control_rows.append({"subtype": subtype, "permutation": i,
                             "p": stats.wilcoxon(did * flips[subtype][i]).pvalue})
control = pd.DataFrame(control_rows)
control.groupby("subtype")["p"].apply(lambda p: (p < 0.05).mean()).rename("rejection rate at 0.05").round(3)


subtype
bradykinetic    0.057
tremor          0.033
Name: rejection rate at 0.05, dtype: float64

## Result 1: false-positive rates


In [8]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


rate_rows = []
for (spec, subtype), block in null.groupby(["spec", "subtype"]):
    k, n = int(block["interval excludes 0"].sum()), len(block)
    low, high = wilson(k, n)
    rate_rows.append({"spec": spec, "subtype": subtype, "null datasets": n, "rejections": k,
                      "false-positive rate": k / n, "95% CI low": low, "95% CI high": high})
for subtype, block in control.groupby("subtype"):
    k, n = int((block["p"] < 0.05).sum()), len(block)
    low, high = wilson(k, n)
    rate_rows.append({"spec": "control: subject-level Wilcoxon", "subtype": subtype, "null datasets": n,
                      "rejections": k, "false-positive rate": k / n, "95% CI low": low, "95% CI high": high})

rates = pd.DataFrame(rate_rows).set_index(["spec", "subtype"])
rates.round(3)


null datasets  rejections  \
spec                            subtype                                   
A: published                    bradykinetic            300         152   
                                tremor                  300         156   
B: first calibration            bradykinetic            300         151   
                                tremor                  300         155   
C: matches MCMC                 bradykinetic            300         153   
                                tremor                  300         171   
control: subject-level Wilcoxon bradykinetic            300          17   
                                tremor                  300          10   

                                              false-positive rate  95% CI low  \
spec                            subtype                                         
A: published                    bradykinetic                0.507       0.450   
                                tremor                      0.520       0.464   
B: first calibration            bradykinetic                0.503       0.447   
                                tremor                      0.517       0.460   
C: matches MCMC                 bradykinetic                0.510       0.454   
                                tremor                      0.570       0.513   
control: subject-level Wilcoxon bradykinetic                0.057       0.036   
                                tremor                      0.033       0.018   

                                              95% CI high  
spec                            subtype                    
A: published                    bradykinetic        0.563  
                                tremor              0.576  
B: first calibration            bradykinetic        0.560  
                                tremor              0.573  
C: matches MCMC                 bradykinetic        0.566  
                                tremor              0.625  
control: subject-level Wilcoxon bradykinetic        0.089  
                                tremor              0.060

## Result 2: posterior width on the real data


In [9]:
mcmc_sd = {}
for subtype in SUBTYPES:
    with np.load(CACHE_DIR / f"{subtype}.npz") as cached:   # read only
        draws = cached["beta"].reshape(-1, len(TERMS))[:, TERMS.index("col:med")]
    mcmc_sd[subtype] = draws.std()

width_rows = []
for spec in SPECS:
    for subtype in SUBTYPES:
        mean, sd = vb_interaction(spec, subtype)
        width_rows.append({"spec": spec, "subtype": subtype, "VB mean": mean, "VB SD": sd,
                           "MCMC SD": mcmc_sd[subtype], "VB / MCMC": sd / mcmc_sd[subtype],
                           "published-style z": mean / sd})

width = pd.DataFrame(width_rows).set_index(["spec", "subtype"])
width.round(4)


VB mean   VB SD  MCMC SD  VB / MCMC  \
spec                 subtype                                             
A: published         tremor         0.4706  0.0814   0.2774     0.2935   
                     bradykinetic   0.0084  0.0853   0.2822     0.3023   
B: first calibration tremor         0.4683  0.0818   0.2774     0.2950   
                     bradykinetic   0.0092  0.0858   0.2822     0.3039   
C: matches MCMC      tremor         0.5294  0.0841   0.2774     0.3032   
                     bradykinetic   0.0446  0.0861   0.2822     0.3050   

                                   published-style z  
spec                 subtype                          
A: published         tremor                   5.7797  
                     bradykinetic             0.0982  
B: first calibration tremor                   5.7226  
                     bradykinetic             0.1077  
C: matches MCMC      tremor                   6.2927  
                     bradykinetic             0.5186

The MCMC SD comes from the reported model, which matches spec C. For specs A and B the ratio compares
the variational SD with the posterior of a slightly different model, so read it as indicative only.

## Summary for the Methods sentence


In [10]:
c_rates = rates.loc["C: matches MCMC"]
c_width = width.loc["C: matches MCMC"]
b_rates = rates.loc["B: first calibration"]
b_width = width.loc["B: first calibration"]

print("Spec C (the model the Methods describes)")
for subtype in SUBTYPES:
    r, w = c_rates.loc[subtype], c_width.loc[subtype]
    print(f"  {subtype:13s} false-positive rate {r['false-positive rate']:.1%} "
          f"(95% CI {r['95% CI low']:.1%} to {r['95% CI high']:.1%}), VB/MCMC SD {w['VB / MCMC']:.2f}")
print("\nSpec B (reproduces the earlier run; previously reported 51.7% and 50.3%, ratio about 0.30)")
for subtype in SUBTYPES:
    r, w = b_rates.loc[subtype], b_width.loc[subtype]
    print(f"  {subtype:13s} false-positive rate {r['false-positive rate']:.1%}, VB/MCMC SD {w['VB / MCMC']:.2f}")


Spec C (the model the Methods describes)
  tremor        false-positive rate 57.0% (95% CI 51.3% to 62.5%), VB/MCMC SD 0.30
  bradykinetic  false-positive rate 51.0% (95% CI 45.4% to 56.6%), VB/MCMC SD 0.31

Spec B (reproduces the earlier run; previously reported 51.7% and 50.3%, ratio about 0.30)
  tremor        false-positive rate 51.7%, VB/MCMC SD 0.29
  bradykinetic  false-positive rate 50.3%, VB/MCMC SD 0.30
